# Create or Update Knowledge Assistant (Invoices)

**Display name** is ``{table_prefix}-invoice-extraction-assistant`` using job parameters from the bundle. **Documents path** is ``{volume}/invoices``.

Use more knowledge assistant functionalities via the REST API https://docs.databricks.com/api/workspace/knowledgeassistants or SDK https://databricks-sdk-py.readthedocs.io/en/latest/workspace/knowledgeassistants/knowledge_assistants.html#databricks.sdk.service.knowledgeassistants.KnowledgeAssistantsAPI

In [0]:
# %pip install databricks-sdk -q

from databricks.sdk import WorkspaceClient
from manage_knowledge_assistant import *

w = WorkspaceClient()

# Defaults match databricks_etl/databricks.yml variables (overridden by job base_parameters)
dbutils.widgets.text("table_prefix", "app")
dbutils.widgets.text("volume", "/Volumes/<CATALOG>/<SCHEMA>/<VOLUME_NAME>")
table_prefix = dbutils.widgets.get("table_prefix")
volume_base = dbutils.widgets.get("volume")

DOCUMENTS_VOLUME_PATH = f"{volume_base.rstrip('/')}/invoices"
DISPLAY_NAME = f"{table_prefix}-invoice-extraction-assistant"

DESCRIPTION = "Answers questions about invoices"
INSTRUCTIONS = "Use the invoices to answer questions about them"

KNOWLEDGE_SOURCE_DISPLAY_NAME = "documents-volume"
KNOWLEDGE_SOURCE_DESCRIPTION = (
    "Documents from the data extraction volume (PDFs and other uploaded files)."
)

print(f"DISPLAY_NAME={DISPLAY_NAME}")
print(f"DOCUMENTS_VOLUME_PATH={DOCUMENTS_VOLUME_PATH}")


In [0]:
ka_id = get_knowledge_assistant_id_by_display_name(w, DISPLAY_NAME)

if ka_id is None:
    response = create_knowledge_assistant(w, DISPLAY_NAME, DESCRIPTION, INSTRUCTIONS)
    ka_id = response.get("id")
    print("Knowledge Assistant created.")
    print(f"id: {ka_id}")
    source_response = create_knowledge_source_files(
        w,
        ka_id,
        display_name=KNOWLEDGE_SOURCE_DISPLAY_NAME,
        description=KNOWLEDGE_SOURCE_DESCRIPTION,
        volume_path=DOCUMENTS_VOLUME_PATH,
    )
    print("Knowledge Source (files) created.")
    print(f"  source id: {source_response.get('id')}")
    status = get_knowledge_assistant(w, ka_id)
    print("Knowledge Assistant state:", status.get("state"))
    source_status = get_knowledge_source(w, ka_id, source_response.get("id"))
    print("Knowledge Source state:", source_status.get("state"))
else:
    print(f"Knowledge Assistant already exists (id={ka_id}). Syncing knowledge sources.")

sync_knowledge_sources(w, ka_id)
print("Knowledge sources sync completed.")
